<a href="https://colab.research.google.com/github/ramyaa-hegde/flyrank-ml-internship-tasks/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramyaa-hegde/flyrank-ml-internship-tasks/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### My baseline action rule

I will rank pseudonymized content items that are in striking distance of a stronger search position: average March position from 4 to 20, with at least 100 March impressions.

For each item, I compare its observed March CTR with the weighted CTR benchmark for its own position bucket. The baseline action score is:

`max(position-bucket benchmark CTR − observed CTR, 0) × March impressions`

This score estimates the observed click gap at the item’s current visibility. It is used to prioritize human review, not to claim that a title, snippet, or content change will cause more clicks.

**Action label:** `REVIEW_CTR_OPPORTUNITY`  
**Reason code:** `STRIKING_DISTANCE_CTR_GAP`

In [5]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE HUGGINGFACE, TOKEN '{hf_token}')"
)

REL = "fact_content_daily_performance"
MARCH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

baseline_df = con.execute(f"""
    WITH content_month AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS march_impressions,
            SUM(gsc_clicks) AS march_clicks,
            SUM(gsc_clicks)::DOUBLE
                / NULLIF(SUM(gsc_impressions), 0) AS march_ctr,
            AVG(NULLIF(gsc_avg_position, 0)) AS march_avg_position,
            COUNT(*) AS observed_days
        FROM read_parquet('{MARCH}')
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100
           AND AVG(NULLIF(gsc_avg_position, 0)) BETWEEN 4 AND 20
    ),
    bucketed AS (
        SELECT *,
            CASE
                WHEN march_avg_position <= 10 THEN '4-10'
                ELSE '11-20'
            END AS position_bucket
        FROM content_month
    ),
    benchmarks AS (
        SELECT
            position_bucket,
            SUM(march_clicks)::DOUBLE
                / NULLIF(SUM(march_impressions), 0) AS benchmark_ctr
        FROM bucketed
        GROUP BY 1
    )
    SELECT
        b.*,
        bm.benchmark_ctr,
        GREATEST(bm.benchmark_ctr - b.march_ctr, 0) AS ctr_gap,
        GREATEST(bm.benchmark_ctr - b.march_ctr, 0)
            * b.march_impressions AS action_score,
        CASE
            WHEN b.march_ctr < bm.benchmark_ctr
                THEN 'STRIKING_DISTANCE_CTR_GAP'
            ELSE 'NO_CTR_GAP'
        END AS reason_code,
        CASE
            WHEN b.march_ctr < bm.benchmark_ctr
                THEN 'REVIEW_CTR_OPPORTUNITY'
            ELSE 'MONITOR'
        END AS action_label
    FROM bucketed AS b
    JOIN benchmarks AS bm
        USING (position_bucket)
    ORDER BY action_score DESC
""").df()

print("=== Baseline action-score input ===")
print(f"One row per client-content item: {len(baseline_df):,}")
print(
    "Items flagged for CTR review: "
    f"{(baseline_df['action_label'] == 'REVIEW_CTR_OPPORTUNITY').sum():,}"
)

display(
    baseline_df[
        [
            "position_bucket",
            "march_impressions",
            "march_ctr",
            "march_avg_position",
            "benchmark_ctr",
            "ctr_gap",
            "action_score",
            "reason_code",
            "action_label",
        ]
    ].head(10)
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Baseline action-score input ===
One row per client-content item: 60,532
Items flagged for CTR review: 40,638


,position_bucket,march_impressions,march_ctr,march_avg_position,benchmark_ctr,ctr_gap,action_score,reason_code,action_label
0,4-10,212404.0,0.000113,7.346909,0.003017,0.002904,616.787418,STRIKING_DISTANCE_CTR_GAP,REVIEW_CTR_OPPORTUNITY
1,4-10,134984.0,0.000007,4.545582,0.003017,0.003009,406.224199,STRIKING_DISTANCE_CTR_GAP,REVIEW_CTR_OPPORTUNITY
2,4-10,124075.0,0.000008,9.385150,0.003017,0.003009,373.313567,STRIKING_DISTANCE_CTR_GAP,REVIEW_CTR_OPPORTUNITY
3,4-10,132593.0,0.000626,5.789019,0.003017,0.002391,317.010951,STRIKING_DISTANCE_CTR_GAP,REVIEW_CTR_OPPORTUNITY
4,4-10,107584.0,0.000139,9.536301,0.003017,0.002877,309.562972,STRIKING_DISTANCE_CTR_GAP,REVIEW_CTR_OPPORTUNITY
5,4-10,89332.0,0.000045,7.786219,0.003017,0.002972,265.499734,STRIKING_DISTANCE_CTR_GAP,REVIEW_CTR_OPPORTUNITY
6,11-20,83834.0,0.000012,11.967474,0.003052,0.003040,254.895082,STRIKING_DISTANCE_CTR_GAP,REVIEW_CTR_OPPORTUNITY
7,4-10,83788.0,0.000072,7.289152,0.003017,0.002945,246.774412,STRIKING_DISTANCE_CTR_GAP,REVIEW_CTR_OPPORTUNITY
8,4-10,82376.0,0.000134,7.794395,0.003017,0.002883,237.514643,STRIKING_DISTANCE_CTR_GAP,REVIEW_CTR_OPPORTUNITY
9,4-10,139417.0,0.001370,5.036458,0.003017,0.001647,229.597820,STRIKING_DISTANCE_CTR_GAP,REVIEW_CTR_OPPORTUNITY


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
from pathlib import Path

# Build and save the complete ranked baseline queue

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

queue_columns = [
    "client_hash_id",
    "content_hash_id",
    "position_bucket",
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_avg_position",
    "observed_days",
    "benchmark_ctr",
    "ctr_gap",
    "action_score",
    "reason_code",
    "action_label",
]

baseline_queue = (
    baseline_df[queue_columns]
    .sort_values(
        by=["action_score", "march_impressions"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

baseline_queue.to_csv(output_path, index=False)

print("=== Ranked queue saved ===")
print(f"Rows written: {len(baseline_queue):,}")
print(f"Flagged review candidates: {(baseline_queue['action_label'] == 'REVIEW_CTR_OPPORTUNITY').sum():,}")
print(f"Saved file: {output_path}")

display(baseline_queue.head(10))

=== Ranked queue saved ===
Rows written: 60,532
Flagged review candidates: 40,638
Saved file: work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,position_bucket,march_impressions,march_clicks,march_ctr,march_avg_position,observed_days,benchmark_ctr,ctr_gap,action_score,reason_code,action_label
0,client_23a62021009f63c4,content_44f34c0a90047651,4-10,212404.0,24.0,0.000113,7.346909,31,0.003017,0.002904,616.787418,STRIKING_DISTANCE_CTR_GAP,REVIEW_CTR_OPPORTUNITY
1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,4-10,134984.0,1.0,0.000007,4.545582,31,0.003017,0.003009,406.224199,STRIKING_DISTANCE_CTR_GAP,REVIEW_CTR_OPPORTUNITY
2,client_73cda7b4e4f265ea,content_fec55986a1868d62,4-10,124075.0,1.0,0.000008,9.385150,31,0.003017,0.003009,373.313567,STRIKING_DISTANCE_CTR_GAP,REVIEW_CTR_OPPORTUNITY
3,client_62f4a7e64f5e0096,content_7c6373141eae744a,4-10,132593.0,83.0,0.000626,5.789019,31,0.003017,0.002391,317.010951,STRIKING_DISTANCE_CTR_GAP,REVIEW_CTR_OPPORTUNITY
4,client_62f4a7e64f5e0096,content_f6116743b00afc2d,4-10,107584.0,15.0,0.000139,9.536301,31,0.003017,0.002877,309.562972,STRIKING_DISTANCE_CTR_GAP,REVIEW_CTR_OPPORTUNITY
5,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,4-10,89332.0,4.0,0.000045,7.786219,31,0.003017,0.002972,265.499734,STRIKING_DISTANCE_CTR_GAP,REVIEW_CTR_OPPORTUNITY
6,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,11-20,83834.0,1.0,0.000012,11.967474,31,0.003052,0.003040,254.895082,STRIKING_DISTANCE_CTR_GAP,REVIEW_CTR_OPPORTUNITY
7,client_a80fca3f171ed1de,content_046fc480045b88f5,4-10,83788.0,6.0,0.000072,7.289152,25,0.003017,0.002945,246.774412,STRIKING_DISTANCE_CTR_GAP,REVIEW_CTR_OPPORTUNITY
8,client_a80fca3f171ed1de,content_9540d884af3e41fd,4-10,82376.0,11.0,0.000134,7.794395,31,0.003017,0.002883,237.514643,STRIKING_DISTANCE_CTR_GAP,REVIEW_CTR_OPPORTUNITY
9,client_73cda7b4e4f265ea,content_f43118e089ecc69a,4-10,139417.0,191.0,0.001370,5.036458,31,0.003017,0.001647,229.597820,STRIKING_DISTANCE_CTR_GAP,REVIEW_CTR_OPPORTUNITY


### Top-20 review approach

The top 20 are the highest-scoring items flagged by the transparent baseline rule. Each row is a pseudonymized client-content item; its action, reason code, evidence note, and failure condition are generated from observed March aggregates.

The score is useful for ordering human review. It is not a prediction of clicks and does not establish why an item’s CTR is below its comparable position-bucket benchmark.

In [7]:
# Top-20 review: real candidates from the saved baseline queue

top_20_review = (
    baseline_queue[
        baseline_queue["action_label"] == "REVIEW_CTR_OPPORTUNITY"
    ]
    .head(20)
    .copy()
)

top_20_review["confidence_note"] = top_20_review.apply(
    lambda row: (
        f"More reliable observed signal: {int(row['observed_days'])} observed days "
        f"and {int(row['march_impressions']):,} impressions."
        if row["observed_days"] >= 28
        else (
            f"Use extra care: only {int(row['observed_days'])} observed days, "
            f"despite {int(row['march_impressions']):,} impressions."
        )
    ),
    axis=1,
)

top_20_review["what_would_make_it_wrong"] = (
    "The CTR gap may reflect query mix, SERP features, brand intent, or measurement "
    "limits rather than a title, snippet, or page-content opportunity."
)

top_20_review = top_20_review.rename(
    columns={"action_label": "action"}
)

print("=== Top-20 baseline action-score review ===")
print(f"Top-20 rows shown: {len(top_20_review)}")

display(
    top_20_review[
        [
            "content_hash_id",
            "client_hash_id",
            "action",
            "reason_code",
            "march_impressions",
            "march_ctr",
            "march_avg_position",
            "action_score",
            "confidence_note",
            "what_would_make_it_wrong",
        ]
    ]
)


=== Top-20 baseline action-score review ===
Top-20 rows shown: 20


,content_hash_id,client_hash_id,action,reason_code,march_impressions,march_ctr,march_avg_position,action_score,confidence_note,what_would_make_it_wrong
0,content_44f34c0a90047651,client_23a62021009f63c4,REVIEW_CTR_OPPORTUNITY,STRIKING_DISTANCE_CTR_GAP,212404.0,0.000113,7.346909,616.787418,More reliable observed signal: 31 observed day...,"The CTR gap may reflect query mix, SERP featur..."
1,content_8e1334d6356668e3,client_73cda7b4e4f265ea,REVIEW_CTR_OPPORTUNITY,STRIKING_DISTANCE_CTR_GAP,134984.0,0.000007,4.545582,406.224199,More reliable observed signal: 31 observed day...,"The CTR gap may reflect query mix, SERP featur..."
2,content_fec55986a1868d62,client_73cda7b4e4f265ea,REVIEW_CTR_OPPORTUNITY,STRIKING_DISTANCE_CTR_GAP,124075.0,0.000008,9.385150,373.313567,More reliable observed signal: 31 observed day...,"The CTR gap may reflect query mix, SERP featur..."
3,content_7c6373141eae744a,client_62f4a7e64f5e0096,REVIEW_CTR_OPPORTUNITY,STRIKING_DISTANCE_CTR_GAP,132593.0,0.000626,5.789019,317.010951,More reliable observed signal: 31 observed day...,"The CTR gap may reflect query mix, SERP featur..."
4,content_f6116743b00afc2d,client_62f4a7e64f5e0096,REVIEW_CTR_OPPORTUNITY,STRIKING_DISTANCE_CTR_GAP,107584.0,0.000139,9.536301,309.562972,More reliable observed signal: 31 observed day...,"The CTR gap may reflect query mix, SERP featur..."
5,content_cd3d932d4e1c8db0,client_9958f0a7ae1df715,REVIEW_CTR_OPPORTUNITY,STRIKING_DISTANCE_CTR_GAP,89332.0,0.000045,7.786219,265.499734,More reliable observed signal: 31 observed day...,"The CTR gap may reflect query mix, SERP featur..."
6,content_9c057b66c30a3abb,client_73cda7b4e4f265ea,REVIEW_CTR_OPPORTUNITY,STRIKING_DISTANCE_CTR_GAP,83834.0,0.000012,11.967474,254.895082,More reliable observed signal: 31 observed day...,"The CTR gap may reflect query mix, SERP featur..."
7,content_046fc480045b88f5,client_a80fca3f171ed1de,REVIEW_CTR_OPPORTUNITY,STRIKING_DISTANCE_CTR_GAP,83788.0,0.000072,7.289152,246.774412,"Use extra care: only 25 observed days, despite...","The CTR gap may reflect query mix, SERP featur..."
8,content_9540d884af3e41fd,client_a80fca3f171ed1de,REVIEW_CTR_OPPORTUNITY,STRIKING_DISTANCE_CTR_GAP,82376.0,0.000134,7.794395,237.514643,More reliable observed signal: 31 observed day...,"The CTR gap may reflect query mix, SERP featur..."
9,content_f43118e089ecc69a,client_73cda7b4e4f265ea,REVIEW_CTR_OPPORTUNITY,STRIKING_DISTANCE_CTR_GAP,139417.0,0.001370,5.036458,229.597820,More reliable observed signal: 31 observed day...,"The CTR gap may reflect query mix, SERP featur..."


### Weak-pick and leakage review

A high score does not make a candidate automatically correct. I treat items with incomplete March observation coverage as weaker picks because their observed CTR gap may not represent a full month. Even fully observed items may be poor recommendations if the difference is driven by query mix, brand intent, or SERP features rather than an on-page CTR opportunity.

The score uses only historical March aggregates: impressions, clicks through observed CTR, and average position. Pseudonymized identifiers are used only to identify rows, while action labels and reason codes are outputs of the rule—not scoring inputs.

In [8]:
# Weak-pick review and leakage audit

# A weaker recommendation has incomplete observation coverage.
weak_picks = top_20_review[
    top_20_review["observed_days"] < 28
].copy()

weak_picks["why_weaker"] = (
    "Fewer than 28 observed March days; interpret the observed CTR gap with extra care."
)

print("=== Weak-pick review ===")
print(f"Top-20 candidates with fewer than 28 observed days: {len(weak_picks)}")

if len(weak_picks) > 0:
    display(
        weak_picks[
            [
                "content_hash_id",
                "client_hash_id",
                "march_impressions",
                "march_ctr",
                "march_avg_position",
                "observed_days",
                "action_score",
                "why_weaker",
            ]
        ]
    )
else:
    print("No top-20 candidates had incomplete observation coverage.")

# Leakage audit: list only the columns used to calculate the baseline score.
score_input_fields = [
    "march_impressions",
    "march_ctr",
    "march_avg_position",
]

output_only_fields = [
    "benchmark_ctr",
    "ctr_gap",
    "action_score",
    "reason_code",
    "action_label",
]

forbidden_terms = [
    "future",
    "next",
    "target",
    "label",
    "product_flag",
    "recommended_action",
]

suspect_score_inputs = [
    field
    for field in score_input_fields
    if any(term in field.lower() for term in forbidden_terms)
]

identifier_fields = ["client_hash_id", "content_hash_id"]
identifiers_used_as_score_inputs = [
    field for field in score_input_fields if field in identifier_fields
]

print("\n=== Leakage audit ===")
print("Source window: March 2026 observed data only.")
print("Score input fields:", score_input_fields)
print("Output-only fields:", output_only_fields)
print("Suspect score-input names:", suspect_score_inputs)
print("Identifiers used as score inputs:", identifiers_used_as_score_inputs)

assert not suspect_score_inputs
assert not identifiers_used_as_score_inputs

print(
    "\nResult: no future-window fields, product flags, target labels, "
    "or identifiers are used as score inputs."
)

=== Weak-pick review ===
Top-20 candidates with fewer than 28 observed days: 1


,content_hash_id,client_hash_id,march_impressions,march_ctr,march_avg_position,observed_days,action_score,why_weaker
7,content_046fc480045b88f5,client_a80fca3f171ed1de,83788.0,0.000072,7.289152,25,246.774412,Fewer than 28 observed March days; interpret t...



=== Leakage audit ===
Source window: March 2026 observed data only.
Score input fields: ['march_impressions', 'march_ctr', 'march_avg_position']
Output-only fields: ['benchmark_ctr', 'ctr_gap', 'action_score', 'reason_code', 'action_label']
Suspect score-input names: []
Identifiers used as score inputs: []

Result: no future-window fields, product flags, target labels, or identifiers are used as score inputs.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.